## 1. Project Overview
This project demonstrates a simple ETL (Extract, Transform, Load) pipeline using real-time weather data from the OpenWeather API. Weather data was collected for Johannesburg, Cape Town, and Durban, then transformed using Pandas and saved as a CSV file for analysis.

## 2. Import Libraries

In [1]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 3. API Configuration

In [2]:
# Load the OpenWeather API key securely
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("OPENWEATHER_API_KEY")

# Set the OpenWeather API URL
url = "https://api.openweathermap.org/data/2.5/weather"

## 4. Extract Weather Data

In [3]:
# Define the cities for weather data collection
cities = ["Johannesburg", "Cape Town", "Durban"]

In [4]:
# Create a function to extract weather data for a city
def get_weather(city):
    params = {
        "q": city,
        "appid": api_key,
        "units": "metric"
    }

    response = requests.get(url, params=params)
    return response.json()

In [5]:
# Extract weather data for all selected cities
weather_data = []

for city in cities:
    weather = get_weather(city)
    weather_data.append(weather)

In [6]:
# Display the raw extracted weather data
print(weather_data)

[{'coord': {'lon': 28.0436, 'lat': -26.2023}, 'weather': [{'id': 804, 'main': 'Clouds', 'description': 'overcast clouds', 'icon': '04d'}], 'base': 'stations', 'main': {'temp': 24.66, 'feels_like': 24.07, 'temp_min': 23.75, 'temp_max': 24.95, 'pressure': 1020, 'humidity': 34, 'sea_level': 1020, 'grnd_level': 836}, 'visibility': 10000, 'wind': {'speed': 1.3, 'deg': 351, 'gust': 1.86}, 'clouds': {'all': 94}, 'dt': 1787754721, 'sys': {'type': 2, 'id': 2005686, 'country': 'ZA', 'sunrise': 1787718421, 'sunset': 1787759571}, 'timezone': 7200, 'id': 993800, 'name': 'Johannesburg', 'cod': 200}, {'coord': {'lon': 18.4232, 'lat': -33.9258}, 'weather': [{'id': 804, 'main': 'Clouds', 'description': 'overcast clouds', 'icon': '04d'}], 'base': 'stations', 'main': {'temp': 24.94, 'feels_like': 24.61, 'temp_min': 24.1, 'temp_max': 25.71, 'pressure': 1016, 'humidity': 43, 'sea_level': 1016, 'grnd_level': 1015}, 'visibility': 10000, 'wind': {'speed': 4.12, 'deg': 180}, 'clouds': {'all': 100}, 'dt': 17877

## 5. Transform Weather Data

In [7]:
# Create a structured dataset from the extracted weather data
weather_cleaned = []

for weather in weather_data:
    weather_cleaned.append({
        "City": weather["name"],
        "Temperature": weather["main"]["temp"],
        "Humidity": weather["main"]["humidity"],
        "Weather Condition": weather["weather"][0]["description"],
        "Wind Speed": weather["wind"]["speed"],
        "Date/Time": pd.to_datetime(weather["dt"], unit="s")
    })

weather_df = pd.DataFrame(weather_cleaned)

print("Cleaned Weather Dataset:")
print(weather_df)

Cleaned Weather Dataset:
           City  Temperature  Humidity Weather Condition  Wind Speed  \
0  Johannesburg        24.66        34   overcast clouds        1.30   
1     Cape Town        24.94        43   overcast clouds        4.12   
2        Durban        21.17        71     broken clouds        0.45   

            Date/Time  
0 2026-08-26 14:32:01  
1 2026-08-26 14:30:31  
2 2026-08-26 14:34:19  


### Data Quality Checks

In [8]:
# Check the number of rows and columns
print("Dataset Shape:")
print(weather_df.shape)

Dataset Shape:
(3, 6)


In [9]:
# Check the data types
print("Data Types:")
print(weather_df.dtypes)

Data Types:
City                           str
Temperature                float64
Humidity                     int64
Weather Condition              str
Wind Speed                 float64
Date/Time            datetime64[s]
dtype: object


In [10]:
# Check for missing values
print("Missing Values:")
print(weather_df.isnull().sum())

Missing Values:
City                 0
Temperature          0
Humidity             0
Weather Condition    0
Wind Speed           0
Date/Time            0
dtype: int64


In [11]:
# Check for duplicate rows
print("Duplicate Rows:")
print(weather_df.duplicated().sum())

Duplicate Rows:
0


In [12]:
# Display basic statistics
print("Basic Statistics:")
print(weather_df[["Temperature", "Humidity", "Wind Speed"]].describe())

Basic Statistics:
       Temperature   Humidity  Wind Speed
count     3.000000   3.000000    3.000000
mean     23.590000  49.333333    1.956667
std       2.100452  19.295941    1.921102
min      21.170000  34.000000    0.450000
25%      22.915000  38.500000    0.875000
50%      24.660000  43.000000    1.300000
75%      24.800000  57.000000    2.710000
max      24.940000  71.000000    4.120000


In [13]:
# Display the final transformed dataset
weather_df

,City,Temperature,Humidity,Weather Condition,Wind Speed,Date/Time
0,Johannesburg,24.66,34,overcast clouds,1.30,2026-08-26 14:32:01
1,Cape Town,24.94,43,overcast clouds,4.12,2026-08-26 14:30:31
2,Durban,21.17,71,broken clouds,0.45,2026-08-26 14:34:19


## 6. Load Weather Data

In [14]:
# Save the transformed weather data as a CSV file
weather_df.to_csv("processed_weather_data.csv", index=False)

print("Weather data saved successfully as processed_weather_data.csv")

Weather data saved successfully as processed_weather_data.csv


## 7. Basic Analysis

In [15]:
# Compare temperatures across cities
print("Temperature Comparison:")
print(weather_df[["City", "Temperature"]])

Temperature Comparison:
           City  Temperature
0  Johannesburg        24.66
1     Cape Town        24.94
2        Durban        21.17


In [16]:
# Identify the city with the highest humidity
highest_humidity = weather_df.loc[weather_df["Humidity"].idxmax()]

print("City with the Highest Humidity:")
print(highest_humidity[["City", "Humidity"]])

City with the Highest Humidity:
City        Durban
Humidity        71
Name: 2, dtype: object


In [17]:
# Compare weather conditions across cities
print("Weather Conditions:")
print(weather_df[["City", "Weather Condition"]])

Weather Conditions:
           City Weather Condition
0  Johannesburg   overcast clouds
1     Cape Town   overcast clouds
2        Durban     broken clouds


In [18]:
# Calculate the average temperature
average_temperature = weather_df["Temperature"].mean()

print("Average Temperature:")
print(f"{average_temperature:.2f}°C")

Average Temperature:
23.59°C


In [19]:
# Compare wind speeds across cities
print("Wind Speed Comparison:")
print(weather_df[["City", "Wind Speed"]])

Wind Speed Comparison:
           City  Wind Speed
0  Johannesburg        1.30
1     Cape Town        4.12
2        Durban        0.45


### Key Findings

- Cape Town had the highest temperature at 24.94°C, while Durban had the lowest temperature at 21.17°C.
- The average temperature across the three cities was 23.59°C.
- Durban had the highest humidity at 71%.
- Johannesburg and Cape Town had overcast clouds, while Durban had broken clouds.
- Cape Town had the highest wind speed at 4.12 m/s, while Durban had the lowest at 0.45 m/s.

## 8. Conclusion

The ETL pipeline successfully extracted real-time weather data from the OpenWeather API for three cities, transformed the data into a structured Pandas DataFrame, and saved the processed data as a CSV file. The analysis provided a simple comparison of temperature, humidity, weather conditions, and wind speed across the cities. This project demonstrated how ETL processes can be used to collect, clean, store, and analyze real-world data.